# SawitCare ML Training Notebook

Colab/VS Code Colab notebook for training the SawitCare ML pipeline.

Models:
- Detector: YOLO11n main model
- Detector baseline: YOLOv8n
- Classifier: EfficientNet B0

Labels:
- Detection: `oil_palm_tree`
- Classification: `healthy`, `suspicious`

Important: SawitCare is for early screening, not final disease diagnosis. `suspicious` means the palm should be inspected in the field.

## 1. Check GPU

In [1]:
!nvidia-smi

/bin/bash: line 1: nvidia-smi: command not found


## 2. Get the repository

If this notebook is already opened inside the cloned repo, this cell will just move to the repo folder. If not, it clones from GitHub.

For a private GitHub repo, use your VS Code Colab extension's GitHub authentication or clone manually with a temporary token.

In [2]:
from pathlib import Path
import os

repo_name = "sawitcare-ml"
repo_url = "https://github.com/Xavrir/sawitcare-ml.git"

if Path.cwd().name == repo_name:
    print(f"Already in repo: {Path.cwd()}")
elif Path('/content/sawitcare-ml').exists():
    os.chdir('/content/sawitcare-ml')
    print(f"Using existing repo: {Path.cwd()}")
else:
    os.chdir('/content')
    !git clone {repo_url}
    os.chdir('/content/sawitcare-ml')
    print(f"Cloned repo: {Path.cwd()}")

Cloning into 'sawitcare-ml'...
remote: Enumerating objects: 38, done.
remote: Counting objects: 100% (38/38), done.
remote: Compressing objects: 100% (33/33), done.
remote: Total 38 (delta 3), reused 38 (delta 3), pack-reused 0 (from 0)
Receiving objects: 100% (38/38), 20.94 KiB | 10.47 MiB/s, done.
Resolving deltas: 100% (3/3), done.
Cloned repo: /content/sawitcare-ml


## 3. Install dependencies

In [3]:
!pip install -q -r requirements.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 6.8 MB/s eta 0:00:0000:0100:01


## 4. Set dataset/API configuration

Do **not** commit API keys to GitHub. Paste the Roboflow key only into the runtime.

In [ ]:
import os

# Paste your Roboflow key here for this runtime only.
# Example: os.environ['ROBOFLOW_API_KEY'] = 'rf_xxx'

os.environ['ROBOFLOW_API_KEY'] = os.environ.get('ROBOFLOW_API_KEY', '')

if not os.environ['ROBOFLOW_API_KEY']:
    print('ROBOFLOW_API_KEY is not set. Detection dataset download will fail until you set it.')
else:
    print('ROBOFLOW_API_KEY is set.')

ROBOFLOW_API_KEY is not set. Detection dataset download will fail until you set it.


## 5. Download and prepare datasets

This will:
- Download classification data from Mendeley
- Download detection data from Roboflow when `ROBOFLOW_API_KEY` is set
- Split/normalize detection data
- Crop classification annotations into `healthy` / `suspicious` folders

In [5]:
!python src/data/setup_datasets.py --roboflow-version 19
!python src/data/split_detection_data.py --raw-dir data/raw/detection --output-dir data/processed/detection
!python src/data/split_classification_data.py --raw-dir data/raw/classification --output-dir data/processed/classification

Extracted data/raw/classification_mendeley.zip -> data/raw/classification
Extracted data/raw/classification/Oil Palm Tree Detection for Anomaly Identification/Oil Palm Tree Detection 4.v15i.tensorflow.zip -> data/raw/classification/Oil Palm Tree Detection for Anomaly Identification/Oil Palm Tree Detection 4.v15i.tensorflow
Now run: python src/data/split_classification_data.py --raw-dir data/raw/classification --output-dir data/processed/classification
Detection dataset needs ROBOFLOW_API_KEY. Wrote instructions: data/raw/detection/README.md
Traceback (most recent call last):
  File "/content/sawitcare-ml/src/data/split_detection_data.py", line 96, in <module>
    main()
  File "/content/sawitcare-ml/src/data/split_detection_data.py", line 91, in main
    pairs = collect_pairs(args.raw_dir)
            ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/sawitcare-ml/src/data/split_detection_data.py", line 52, in collect_pairs
    raise ValueError(f"No image/label pairs found in {raw_dir}")
Val

## 6. Fix YOLO dataset path for Colab

The local repo may contain an absolute path from your laptop. This cell rewrites it to the current runtime path.

In [6]:
from pathlib import Path
import yaml

cfg_path = Path('configs/detection_data.yaml')
cfg = yaml.safe_load(cfg_path.read_text())
cfg['path'] = str((Path.cwd() / 'data/processed/detection').resolve())
cfg_path.write_text(yaml.safe_dump(cfg, sort_keys=False))
print(cfg_path.read_text())

path: /content/sawitcare-ml/data/processed/detection
train: images/train
val: images/val
test: images/test
names:
  0: oil_palm_tree



## 7. Sanity-check dataset counts

In [7]:
from pathlib import Path

for split in ['train', 'val', 'test']:
    image_count = len(list((Path('data/processed/detection/images') / split).glob('*')))
    label_count = len(list((Path('data/processed/detection/labels') / split).glob('*.txt')))
    print('detection', split, image_count, 'images', label_count, 'labels')

for split in ['train', 'val', 'test']:
    healthy = len(list((Path('data/processed/classification') / split / 'healthy').glob('*')))
    suspicious = len(list((Path('data/processed/classification') / split / 'suspicious').glob('*')))
    print('classification', split, {'healthy': healthy, 'suspicious': suspicious})

detection train 0 images 0 labels
detection val 0 images 0 labels
detection test 0 images 0 labels
classification train {'healthy': 217, 'suspicious': 166}
classification val {'healthy': 46, 'suspicious': 35}
classification test {'healthy': 48, 'suspicious': 37}


## 8. Quick smoke test training

Run a short training first to catch path/GPU errors before spending hours on full training.

In [8]:
!python src/training/train_detector_yolo.py \
  --model yolo11n.pt \
  --data configs/detection_data.yaml \
  --imgsz 640 \
  --epochs 3 \
  --batch 8 \
  --device 0

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
CUDA is not available. Falling back to CPU.
Ultralytics 8.4.51 🚀 Python-3.12.13 torch-2.10.0+cpu CPU (Intel Xeon CPU @ 2.20GHz)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=configs/detection_data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=3, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=Fals

## 9. Train YOLO11n main detector

Run this after the smoke test works.

In [9]:
!python src/training/train_detector_yolo.py \
  --model yolo11n.pt \
  --data configs/detection_data.yaml \
  --imgsz 640 \
  --epochs 100 \
  --batch 8 \
  --device 0

CUDA is not available. Falling back to CPU.
Ultralytics 8.4.51 🚀 Python-3.12.13 torch-2.10.0+cpu CPU (Intel Xeon CPU @ 2.20GHz)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=configs/detection_data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolo11n, nbs=64, nms=False, opset=None, optimize=False, optim

## 10. Train YOLOv8n baseline detector

In [10]:
!python src/training/train_detector_yolo.py \
  --model yolov8n.pt \
  --data configs/detection_data.yaml \
  --imgsz 640 \
  --epochs 100 \
  --batch 8 \
  --device 0

CUDA is not available. Falling back to CPU.
Ultralytics 8.4.51 🚀 Python-3.12.13 torch-2.10.0+cpu CPU (Intel Xeon CPU @ 2.20GHz)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=configs/detection_data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolov8n, nbs=64, nms=False, opset=None, optimize=False, optim

## 11. Train EfficientNet B0 classifier

In [ ]:
!python src/training/train_classifier.py \
  --data data/processed/classification \
  --epochs 50 \
  --batch 32 \
  --device cuda

CUDA is not available. Falling back to CPU.
Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth
100% 20.5M/20.5M [00:00<00:00, 122MB/s] 
Epoch 1/50 train_loss=0.6703 val_loss=0.6334 val_f1=0.6303
Saved best classifier: models/classifier/efficientnet_b0_best.pt
Epoch 2/50 train_loss=0.5968 val_loss=0.5643 val_f1=0.7209
Saved best classifier: models/classifier/efficientnet_b0_best.pt
train:  17% 2/12 [00:08<00:39,  3.99s/it]

## 12. Evaluate models

In [ ]:
!python src/evaluation/eval_detector.py \
  --model models/detector/yolo11n_best.pt \
  --data configs/detection_data.yaml \
  --device 0

!python src/evaluation/eval_classifier.py \
  --model models/classifier/efficientnet_b0_best.pt \
  --data data/processed/classification \
  --device cuda

## 13. Save trained weights

Option A downloads directly. Option B copies to Google Drive if mounted.

In [ ]:
from pathlib import Path

for path in [
    Path('models/detector/yolo11n_best.pt'),
    Path('models/detector/yolov8n_best.pt'),
    Path('models/classifier/efficientnet_b0_best.pt'),
]:
    print(path, 'exists=' + str(path.exists()))

In [ ]:
# Direct download in Google Colab. Skip if your VS Code Colab extension does not support google.colab.
try:
    from google.colab import files
    for path in [
        'models/detector/yolo11n_best.pt',
        'models/detector/yolov8n_best.pt',
        'models/classifier/efficientnet_b0_best.pt',
    ]:
        if Path(path).exists():
            files.download(path)
except Exception as exc:
    print('Direct Colab download skipped:', exc)

In [ ]:
# Optional: save to Google Drive.
# from google.colab import drive
# drive.mount('/content/drive')
# !mkdir -p /content/drive/MyDrive/sawitcare-ml/models
# !cp -v models/detector/*_best.pt /content/drive/MyDrive/sawitcare-ml/models/ 2>/dev/null || true
# !cp -v models/classifier/*_best.pt /content/drive/MyDrive/sawitcare-ml/models/ 2>/dev/null || true